# Step 6 — Track Experiments with DVC

En este notebook se prueban diferentes configuraciones del modelo usando `params.yaml` y `dvc repro`.

El objetivo es validar que podemos cambiar hiperparámetros sin modificar código, reproducir el pipeline y comparar métricas.cada script.

# Imports

In [4]:
#Imports
from pathlib import Path
import subprocess
import yaml
import json
import copy
import pandas as pd

# Config

In [8]:
#Config
PROJECT_ROOT = Path.cwd()
PARAMS_PATH = PROJECT_ROOT / "params.yaml"
METRICS_PATH = PROJECT_ROOT / "reports/metrics.json"

with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    baseline_params = yaml.safe_load(f)

baseline_params["train"]

{'model_type': 'ridge',
 'scoring': 'r2',
 'cv_splits': 10,
 'cv_random_state': 42,
 'n_jobs': -1,
 'alphas': [1000, 500, 200, 100, 50, 20, 10, 1, 0.1, 0.01]}

## 1. Helper para comandos

In [10]:
#HELPER FUNCTION
def run_command(command):
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        shell=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Falló el comando: {command}")

    return result


def save_params(params):
    with open(PARAMS_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(params, f, sort_keys=False, allow_unicode=True)


def load_metrics():
    with open(METRICS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

## 1. Métricas baseline

In [13]:
run_command("dvc repro")

baseline_metrics = load_metrics()
baseline_metrics

Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.



{'R2_train': 0.8722779520936734,
 'R2_test': 0.8586681107871221,
 'RMSE_train_pesos': 820764.7607973475,
 'RMSE_test_pesos': 724228.728716661,
 'MAE_train_pesos': 598614.5921648004,
 'MAE_test_pesos': 549476.6093178215}

## 2. Ejecutar experimentos

In [16]:
experiments = [
    {"experiment": "baseline_grid", "alphas": baseline_params["train"]["alphas"]},
    {"experiment": "ridge_alpha_1", "alphas": [1]},
    {"experiment": "ridge_alpha_10", "alphas": [10]},
    {"experiment": "ridge_alpha_20", "alphas": [20]},
    {"experiment": "ridge_alpha_50", "alphas": [50]},
    {"experiment": "ridge_alpha_100", "alphas": [100]},
]

results = []

for exp in experiments:
    print("=" * 80)
    print(f"Running experiment: {exp['experiment']}")
    print("=" * 80)

    params = copy.deepcopy(baseline_params)
    params["train"]["alphas"] = exp["alphas"]

    save_params(params)

    run_command("dvc repro")

    metrics = load_metrics()

    results.append({
        "experiment": exp["experiment"],
        "alphas": exp["alphas"],
        "R2_train": metrics["R2_train"],
        "R2_test": metrics["R2_test"],
        "RMSE_test_pesos": metrics["RMSE_test_pesos"],
        "MAE_test_pesos": metrics["MAE_test_pesos"]
    })

results_df = pd.DataFrame(results)
results_df

Running experiment: baseline_grid
Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.

Running experiment: ridge_alpha_1
Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Running stage 'train':
> python src/stages/train.py
Train data loaded from: data/processed/train_data.csv
Train data shape: (428, 18)
Modelo entrenado correctamente.
Mejores hiperparámetros: {'alpha': 1}
R2 CV train: 0.8517684089549741
Modelo guardado en: models/modelo_final.pkl
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python src/stages/evaluate.py
Train data loaded from: data/processed/train_data.csv
Test data loaded from: data/processed/test_data.csv
Train shape: (428, 18)
Test shape: (108, 18)
Model loaded from: models/modelo_final.pkl
Scaler Y loaded from: models/scaler_Y.pkl
Metrics:
{'R2_train': 0.874751595232

,experiment,alphas,R2_train,R2_test,RMSE_test_pesos,MAE_test_pesos
0,baseline_grid,"[1000, 500, 200, 100, 50, 20, 10, 1, 0.1, 0.01]",0.872278,0.858668,724228.728717,549476.609318
1,ridge_alpha_1,[1],0.874752,0.850296,745370.549050,567975.076364
2,ridge_alpha_10,[10],0.874044,0.855197,733068.753006,556810.675649
3,ridge_alpha_20,[20],0.872278,0.858668,724228.728717,549476.609318
4,ridge_alpha_50,[50],0.863808,0.861804,716149.511711,542333.890724
5,ridge_alpha_100,[100],0.846097,0.855988,731064.502927,549134.305230


## 3. Comparar experimentos

In [19]:
results_df.sort_values("R2_test", ascending=False)

,experiment,alphas,R2_train,R2_test,RMSE_test_pesos,MAE_test_pesos
4,ridge_alpha_50,[50],0.863808,0.861804,716149.511711,542333.890724
0,baseline_grid,"[1000, 500, 200, 100, 50, 20, 10, 1, 0.1, 0.01]",0.872278,0.858668,724228.728717,549476.609318
3,ridge_alpha_20,[20],0.872278,0.858668,724228.728717,549476.609318
5,ridge_alpha_100,[100],0.846097,0.855988,731064.502927,549134.305230
2,ridge_alpha_10,[10],0.874044,0.855197,733068.753006,556810.675649
1,ridge_alpha_1,[1],0.874752,0.850296,745370.549050,567975.076364


In [21]:
results_df.sort_values("RMSE_test_pesos", ascending=True)

,experiment,alphas,R2_train,R2_test,RMSE_test_pesos,MAE_test_pesos
4,ridge_alpha_50,[50],0.863808,0.861804,716149.511711,542333.890724
0,baseline_grid,"[1000, 500, 200, 100, 50, 20, 10, 1, 0.1, 0.01]",0.872278,0.858668,724228.728717,549476.609318
3,ridge_alpha_20,[20],0.872278,0.858668,724228.728717,549476.609318
5,ridge_alpha_100,[100],0.846097,0.855988,731064.502927,549134.305230
2,ridge_alpha_10,[10],0.874044,0.855197,733068.753006,556810.675649
1,ridge_alpha_1,[1],0.874752,0.850296,745370.549050,567975.076364


## 4. Restaurar configuración baseline

In [24]:
save_params(baseline_params)

run_command("dvc repro")

restored_metrics = load_metrics()
restored_metrics

Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

Stage 'evaluate' is cached - skipping run, checking out outputs
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add 'reports\.gitignore' dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.



{'R2_train': 0.8722779520936734,
 'R2_test': 0.8586681107871221,
 'RMSE_train_pesos': 820764.7607973475,
 'RMSE_test_pesos': 724228.728716661,
 'MAE_train_pesos': 598614.5921648004,
 'MAE_test_pesos': 549476.6093178215}

## 5. Validación

In [36]:
expected_metrics = {
    "R2_test": 0.8586681107871221,
    "RMSE_test_pesos": 724228.7287166608,
    "MAE_test_pesos": 549476.6093178215,
}

for metric_name, expected_value in expected_metrics.items():
    current_value = restored_metrics[metric_name]
    diff = abs(current_value - expected_value)

    print(metric_name)
    print("Esperado:", expected_value)
    print("Actual:  ", current_value)
    print("Dif:     ", diff)
    print()

assert abs(restored_metrics["R2_test"] - expected_metrics["R2_test"]) < 1e-8
assert abs(restored_metrics["RMSE_test_pesos"] - expected_metrics["RMSE_test_pesos"]) < 1e-2
assert abs(restored_metrics["MAE_test_pesos"] - expected_metrics["MAE_test_pesos"]) < 1e-2

print("Configuración baseline restaurada correctamente.")

R2_test
Esperado: 0.8586681107871221
Actual:   0.8586681107871221
Dif:      0.0

RMSE_test_pesos
Esperado: 724228.7287166608
Actual:   724228.728716661
Dif:      2.3283064365386963e-10

MAE_test_pesos
Esperado: 549476.6093178215
Actual:   549476.6093178215
Dif:      0.0

Configuración baseline restaurada correctamente.


## Conclusión

Se probaron diferentes configuraciones de Ridge usando `params.yaml` y `dvc repro`.

Este paso demuestra que el pipeline permite experimentar sin modificar código, manteniendo trazabilidad y reproducibilidad.